# NF-v3 temporal graph construction

This notebook is intentionally thin. The tested graph builder lives in `code/python/scripts/build_nfv3_graphs.py`; this notebook only mounts Drive, configures paths, runs a small smoke build, and inspects its audit.

Run the smoke build first. Do not run the full build until its generated schema, mappings, provenance, and audit have been reviewed.

In [1]:

# ============================================================
# SETUP - Run this cell first
# ============================================================
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path('/content/temporalgnn-nids')
REPO_URL = 'https://github.com/tatipar/temporalgnn-nids.git'
BRANCH = 'feat/fair-retrain-clean'

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)], check=True)

sys.path.append(str(REPO_ROOT / 'code/python'))

In [2]:
!pip install -q torch-geometric

from google.colab import drive
drive.mount('/content/drive')

import json

PROJECT_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain')
CORRECTED_ROOT = PROJECT_ROOT / 'corrected_data' / 'infiltration_v1'
CORRECTED_CSV = CORRECTED_ROOT / 'nfv3_corrected.csv'
CORRECTED_MANIFEST = CORRECTED_ROOT / 'nfv3_corrected.manifest.json'

assert REPO_ROOT.is_dir(), f'Repository not found: {REPO_ROOT}'
assert CORRECTED_CSV.is_file(), f'Corrected CSV not found: {CORRECTED_CSV}'
assert CORRECTED_MANIFEST.is_file(), f'Corrected manifest not found: {CORRECTED_MANIFEST}'

GRAPH_VERSION = 'infiltration_v1_w30_tcpflags_v1'
PREFLIGHT_ROOT = PROJECT_ROOT / 'graphs' / f'{GRAPH_VERSION}_preflight'
SMOKE_ROOT = PROJECT_ROOT / 'graphs' / f'{GRAPH_VERSION}_smoke'
FULL_ROOT = PROJECT_ROOT / 'graphs' / GRAPH_VERSION
PROFILES = ('nfv3_extended', 'portable_core')

print(f'Corrected CSV: {CORRECTED_CSV}')
print(f'Preflight output: {PREFLIGHT_ROOT}')
print(f'Smoke output: {SMOKE_ROOT}')
print(f'Full output: {FULL_ROOT}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00
Mounted at /content/drive
Corrected CSV: /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv
Preflight output: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1_preflight
Smoke output: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1_smoke
Full output: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1


In [3]:
%cd /content/temporalgnn-nids/code/python
!python -m unittest discover -s tests -p 'test_*.py' -v
%cd /content/temporalgnn-nids

/content/temporalgnn-nids/code/python
test_rejects_flow_start_regression_between_chunks (test_build_nfv3_graphs.CompleteWindowIteratorTests.test_rejects_flow_start_regression_between_chunks) ... ok
test_varied_durations_remain_ordered_across_chunk_boundaries (test_build_nfv3_graphs.CompleteWindowIteratorTests.test_varied_durations_remain_ordered_across_chunk_boundaries) ... ok
test_checkpoint_publishes_mapping_before_state (test_build_nfv3_graphs.ResumeMappingTests.test_checkpoint_publishes_mapping_before_state) ... ok
test_replay_uses_sources_then_destinations_like_graph_building (test_build_nfv3_graphs.ResumeMappingTests.test_replay_uses_sources_then_destinations_like_graph_building) ... ok
test_resume_replay_matches_uninterrupted_mapping (test_build_nfv3_graphs.ResumeMappingTests.test_resume_replay_matches_uninterrupted_mapping) ... ok
test_every_port_and_protocol_gets_one_category (test_graph_construction.GraphSchemaTests.test_every_port_and_protocol_gets_one_category) ... ok
test_

In [4]:
preflight_command = [
    'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
    '--input-csv', str(CORRECTED_CSV),
    '--corrected-manifest', str(CORRECTED_MANIFEST),
    '--output-root', str(PREFLIGHT_ROOT),
    '--chunksize', '250000',
    '--preflight-only',
    '--overwrite',
]
for profile in PROFILES:
    preflight_command += ['--profile', profile]

print(' '.join(preflight_command))
subprocess.run(preflight_command, check=True)

preflight = json.loads((PREFLIGHT_ROOT / 'feature_preflight.json').read_text())
print('Preflight status:', preflight['status'])
print('Input rows / positives:', preflight['input_rows'], preflight['positive_rows'])
print('Retained rows / positives:', preflight['retained_rows'], preflight['retained_positive_rows'])
print('Excluded rows / positives:', preflight['excluded_rows'], preflight['excluded_positive_rows'])
print('Counts by source file:', json.dumps(preflight['by_source_file'], indent=2))
print('Invalid endpoint rows (union):', preflight['invalid_any_endpoint_rows'])
print('Invalid endpoint positive rows:', preflight['invalid_any_endpoint_positive_rows'])
print('Invalid source endpoint rows:', preflight['invalid_source_endpoint_rows'])
print('Invalid destination endpoint rows:', preflight['invalid_destination_endpoint_rows'])
print('Invalid source endpoint reasons:', preflight['invalid_source_endpoint_reasons'])
print('Invalid destination endpoint reasons:', preflight['invalid_destination_endpoint_reasons'])
print('Invalid ports:', preflight['invalid_port_rows'])
print('Invalid protocols:', preflight['invalid_protocol_rows'])
print('Invalid TCP flag bitmasks:', preflight['invalid_tcp_flags_rows'])
print('Non-TCP rows with non-zero TCP flags:', preflight['non_tcp_nonzero_tcp_flags_rows'])
print('Invalid binary targets:', preflight['invalid_binary_target_rows'])
print('Invalid time/duration rows:', preflight['invalid_time_or_duration_rows'])
print('Flow-end differences >1 ms:', preflight['flow_end_difference_gt_1ms_rows'])
print('Maximum absolute flow-end difference (ms):', preflight['flow_end_max_absolute_difference_ms'])
print('Invalid numeric rows:', preflight['invalid_numeric_rows_by_profile'])
print('Port categories:', preflight['port_category_counts'])
print('Port zero by protocol:', preflight['port_zero_by_protocol'])
print('Top other privileged ports:', preflight['other_privileged_top_ports'])
print('Top other high ports:', preflight['other_high_top_ports'])

python /content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py --input-csv /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv --corrected-manifest /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json --output-root /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1_preflight --chunksize 250000 --preflight-only --overwrite --profile nfv3_extended --profile portable_core
Preflight status: passed
Input rows / positives: 4180260 112052
Retained rows / positives: 4157964 112052
Excluded rows / positives: 22296 0
Counts by source file: {
  "cicids2018v3_thu0103.csv": {
    "excluded_positive_rows": 0,
    "excluded_rows": 11576,
    "input_rows": 2147533,
    "invalid_binary_target_rows": 0,
    "invalid_endpoint_positive_rows": 0,
    "invalid_endpoint_rows": 11576,
    "invalid_time_or_duration_positive_rows": 0,
    "invalid_time_or_duration_rows": 0,
    "positiv

In [5]:
command = [
    'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
    '--input-csv', str(CORRECTED_CSV),
    '--corrected-manifest', str(CORRECTED_MANIFEST),
    '--output-root', str(SMOKE_ROOT),
    '--chunksize', '250000',
    '--max-windows', '10',
    '--overwrite',
]
for profile in PROFILES:
    command += ['--profile', profile]

print(' '.join(command))
subprocess.run(command, check=True)

python /content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py --input-csv /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv --corrected-manifest /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json --output-root /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1_smoke --chunksize 250000 --max-windows 10 --overwrite --profile nfv3_extended --profile portable_core


CompletedProcess(args=['python', '/content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py', '--input-csv', '/content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv', '--corrected-manifest', '/content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json', '--output-root', '/content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1_smoke', '--chunksize', '250000', '--max-windows', '10', '--overwrite', '--profile', 'nfv3_extended', '--profile', 'portable_core'], returncode=0)

In [6]:
audit = json.loads((SMOKE_ROOT / 'graph_audit.json').read_text())
configuration = json.loads((SMOKE_ROOT / 'build_configuration.json').read_text())

print('Audit status:', audit['status'])
print('Day-1 cutoffs:', configuration['day1_cutoffs'])
print('Day-1 decision-time distribution:')
print(json.dumps(configuration['day1_cutoffs']['decision_time_distribution'], indent=2))
print('Profile schema hashes:')
for name, digest in configuration['profiles'].items():
    print(f'- {name}: {digest}')

for day_name in ('day1', 'day2'):
    mapping = json.loads((SMOKE_ROOT / 'mappings' / f'{day_name}_ip_to_id.json').read_text())
    print(f"{day_name}: {mapping['entries']} mapped IPs; policy={mapping['creation_policy']}")

print('Artifact checksum summary:')
print(json.dumps(audit['artifacts'], indent=2))

Audit status: partial
Day-1 cutoffs: {'raw_train_end_ms': 1519836776999, 'raw_val_end_ms': 1519849633500, 'train_end_ms': 1519836780000, 'val_end_ms': 1519849650000}
Profile schema hashes:
- nfv3_extended: 30e214b162eecc4c2578c658a5711534d22913d2279cb032429cc38df6c224c3
- portable_core: 95bf10470b10ba9039ce0753881f4743809dd583b8c04e8b7dc1c72706b76ae5
day1: 27 mapped IPs; policy=append_only_first_valid_chronological_appearance
day2: 35 mapped IPs; policy=append_only_first_valid_chronological_appearance


In [7]:
import pandas as pd
import torch

sample_provenance = next((SMOKE_ROOT / 'provenance' / 'day1').glob('graph_*.csv'))
sample_graph = SMOKE_ROOT / 'nfv3_extended' / 'train' / f'{sample_provenance.stem}.pt'
provenance = pd.read_csv(sample_provenance)
graph = torch.load(sample_graph, weights_only=False)

print('Graph:', sample_graph.name)
print('Nodes:', graph.num_nodes)
print('Edges:', graph.edge_index.shape[1])
print('Edge feature shape:', tuple(graph.edge_attr.shape))
print('Decision time (UTC epoch ms):', graph.timestamp)
display(provenance.head())

Graph: graph_1519776780000.pt
Nodes: 2
Edges: 1
Edge feature shape: (1, 40)
Decision time (UTC epoch ms): 1519776780000


,flow_id,source_file,source_row_id,source_ip,destination_ip,source_global_id,destination_global_id,edge_position,flow_start_ms,flow_end_ms,decision_time_ms,window_start_ms,window_end_ms,window_wait_ms,split,binary_target
0,cicids2018v3_wed2802.csv:0,cicids2018v3_wed2802.csv,0,5.188.9.25,172.31.64.89,0,1,0,1.519777e+12,1.519777e+12,1519776780000,1519776750000,1519776780000,4146.0,train,0


In [8]:
from pathlib import Path
import json
import shutil

### 1. Prepare the simulation

SOURCE_SMOKE = (
    PROJECT_ROOT / "graphs"
    / "infiltration_v1_w30_tcpflags_v1_smoke"
)

RESUME_TEST_ROOT = Path("/content/nfv3_resume_test")
REFERENCE_ROOT = Path("/content/nfv3_reference_15")

for path in (RESUME_TEST_ROOT, REFERENCE_ROOT):
    if path.exists():
        shutil.rmtree(path)

shutil.copytree(SOURCE_SMOKE, RESUME_TEST_ROOT)

state_path = RESUME_TEST_ROOT / "build_state.json"
state = json.loads(state_path.read_text())

day1_graphs = sorted(
    (RESUME_TEST_ROOT / "nfv3_extended" / "train").glob("graph_*.pt")
)
day2_graphs = sorted(
    (RESUME_TEST_ROOT / "nfv3_extended" / "test2").glob("graph_*.pt")
)

assert len(day1_graphs) == 10
assert len(day2_graphs) == 10

day1_fifth = int(day1_graphs[4].stem.split("_")[1])
day2_fifth = int(day2_graphs[4].stem.split("_")[1])

state["days"]["day1"]["last_completed_decision_time_ms"] = day1_fifth
state["days"]["day2"]["last_completed_decision_time_ms"] = day2_fifth
state["completed"] = False

state_path.write_text(json.dumps(state, indent=2, sort_keys=True) + "\n")

print("Simulated checkpoint:")
print("day1:", day1_fifth)
print("day2:", day2_fifth)

### 2. Resume from window 5 to 15

resume_command = [
    "python",
    str(REPO_ROOT / "code/python/scripts/build_nfv3_graphs.py"),
    "--input-csv", str(CORRECTED_CSV),
    "--corrected-manifest", str(CORRECTED_MANIFEST),
    "--output-root", str(RESUME_TEST_ROOT),
    "--chunksize", "250000",
    "--max-windows", "10",
    "--resume",
]

for profile in PROFILES:
    resume_command += ["--profile", profile]

print(" ".join(resume_command))
subprocess.run(resume_command, check=True)

### 3. Build the continuous reference of 15 windows

reference_command = [
    "python",
    str(REPO_ROOT / "code/python/scripts/build_nfv3_graphs.py"),
    "--input-csv", str(CORRECTED_CSV),
    "--corrected-manifest", str(CORRECTED_MANIFEST),
    "--output-root", str(REFERENCE_ROOT),
    "--chunksize", "250000",
    "--max-windows", "15",
    "--overwrite",
]

for profile in PROFILES:
    reference_command += ["--profile", profile]

print(" ".join(reference_command))
subprocess.run(reference_command, check=True)

### 4. Compare maps, graphs, and provenance

import pandas as pd
import torch

for day in ("day1", "day2"):
    resumed_map = json.loads(
        (RESUME_TEST_ROOT / "mappings" / f"{day}_ip_to_id.json").read_text()
    )
    reference_map = json.loads(
        (REFERENCE_ROOT / "mappings" / f"{day}_ip_to_id.json").read_text()
    )
    assert resumed_map == reference_map, f"Mapping mismatch: {day}"

resumed_graphs = sorted(
    path.relative_to(RESUME_TEST_ROOT)
    for path in RESUME_TEST_ROOT.rglob("graph_*.pt")
)
reference_graphs = sorted(
    path.relative_to(REFERENCE_ROOT)
    for path in REFERENCE_ROOT.rglob("graph_*.pt")
)

assert resumed_graphs == reference_graphs

tensor_fields = (
    "edge_index",
    "edge_attr",
    "y",
    "global_node_ids",
)

metadata_fields = (
    "num_nodes",
    "timestamp",
    "window_start",
    "window_end",
    "feature_profile",
    "schema_hash",
)

for relative_path in resumed_graphs:
    resumed = torch.load(
        RESUME_TEST_ROOT / relative_path,
        weights_only=False,
    )
    reference = torch.load(
        REFERENCE_ROOT / relative_path,
        weights_only=False,
    )

    for field in tensor_fields:
        torch.testing.assert_close(
            getattr(resumed, field),
            getattr(reference, field),
            rtol=0,
            atol=0,
        )

    for field in metadata_fields:
        assert getattr(resumed, field) == getattr(reference, field)

resumed_provenance = sorted(
    path.relative_to(RESUME_TEST_ROOT)
    for path in (RESUME_TEST_ROOT / "provenance").rglob("graph_*.csv")
)
reference_provenance = sorted(
    path.relative_to(REFERENCE_ROOT)
    for path in (REFERENCE_ROOT / "provenance").rglob("graph_*.csv")
)

assert resumed_provenance == reference_provenance

for relative_path in resumed_provenance:
    resumed = pd.read_csv(RESUME_TEST_ROOT / relative_path)
    reference = pd.read_csv(REFERENCE_ROOT / relative_path)
    pd.testing.assert_frame_equal(resumed, reference)

print("Resume equivalence: passed")
print("Graphs compared:", len(resumed_graphs))
print("Provenance files compared:", len(resumed_provenance))


Simulated checkpoint:
day1: 1519776900000
day2: 1519862550000
python /content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py --input-csv /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv --corrected-manifest /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json --output-root /content/nfv3_resume_test --chunksize 250000 --max-windows 10 --resume --profile nfv3_extended --profile portable_core
python /content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py --input-csv /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv --corrected-manifest /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json --output-root /content/nfv3_reference_15 --chunksize 250000 --max-windows 15 --overwrite --profile nfv3_extended --profile portable_core
Resume equivalence: passed
Graphs compared: 60
Provenance files compar

## Full build

Run this cell only after reviewing the smoke-build output. Use a new versioned output directory instead of overwriting a reviewed graph collection. If Colab disconnects, rerun the same command with `--resume`.

In [ ]:
# full_command = [
#     'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
#     '--input-csv', str(CORRECTED_CSV),
#     '--corrected-manifest', str(CORRECTED_MANIFEST),
#     '--output-root', str(FULL_ROOT),
#     '--chunksize', '250000',
# ]
# for profile in PROFILES:
#     full_command += ['--profile', profile]
# subprocess.run(full_command, check=True)

# To resume an interrupted full build, append '--resume' to full_command and run it again.